# TD(0) prediction with terminal and truncated episodes

Estimate a policy's state values from recorded episodes with TD(0) (u20-l04). The episodes were generated by the policy already; you only replay them and update a value table.

```python
def td0_prediction(episodes, V0, alpha, gamma):
    ...
    return V
```

- `episodes`: a list of episodes. Each episode is a list of transitions `(s, r, s2, terminated)` in time order: `s` and `s2` are ints (the state before and after the step), `r` is the float reward, and `terminated` is a bool, `True` when `s2` is a terminal state.
- `V0`: a float array of shape `(n_states,)`, the starting value table. Its entries for terminal states may be **nonzero** (for example an optimistic initialization); they must not affect any target.
- `alpha`: the step size. `gamma`: the discount factor.

Process the episodes in order and the transitions of each episode in order. For each transition, update the table immediately (the next transition's target uses the updated table):

$$V(s) \leftarrow V(s) + \alpha\big(\text{target} - V(s)\big), \qquad \text{target} = \begin{cases} r & \text{if terminated} \\ r + \gamma V(s_2) & \text{otherwise.}\end{cases}$$

Some episodes were cut by a time limit: their last transition has `terminated = False`, and it still bootstraps from $V(s_2)$, as with Gymnasium's `truncated` flag. Return the final table as a new float array of shape `(n_states,)`. Do not modify `V0` or `episodes`.

**Example.** States 0 to 3, state 3 terminal, `alpha = 0.5`, `gamma = 0.9`, `V0 = np.array([0.0, 0.0, 0.0, 5.0])`:

```python
episodes = [
    [(0, -1.0, 1, False), (1, -1.0, 2, False), (2, 10.0, 3, True)],   # reaches the terminal state
    [(0, -1.0, 1, False), (1, -1.0, 2, False)],                       # cut by a time limit
]
```

After the first episode the table is `[-0.5, -0.5, 5.0, 5.0]` (the last target is $10$, not $10 + 0.9 \cdot 5$). The function returns `[-0.975, 1.5, 5.0, 5.0]`: in the second episode the last step bootstraps, so state 1's target is $-1 + 0.9 \cdot 5 = 3.5$.

**Restrictions.** Only `numpy`.

**Constraints.** Up to 1000 episodes and $10^5$ transitions in total; `n_states` $\le 1000$.


## How to use this notebook

1. Write your solution in the next cell. Running it saves the file `problem4_submission.py`, the same way the contest's Colab notebook does.
2. Run the grader cell at the bottom. It runs the example and the hidden tests and checks the import and ban rules.
3. Edit, re-run the solution cell, then re-run the grader.

From the USAAIO Prep study site (https://bengreff.github.io/usaaio-prep/); not affiliated with or endorsed by USAAIO.

In [ ]:
%%writefile problem4_submission.py
import numpy as np


def td0_prediction(episodes, V0, alpha, gamma):
    # Your code
    return V


Optional: import it the way the grader does and try the example from the statement.

In [ ]:
import importlib
import problem4_submission as submission
importlib.reload(submission)
# e.g. submission.td0_prediction(...)

## Grade

In [ ]:
#@title Grader: run this cell to grade your solution { display-mode: "form" }
# The grading code is hidden on purpose (double-click to see it). It writes the site's test harness and this
# problem's grader to a temporary folder, imports your file the way the contest grader does, checks the import
# and ban rules, runs every test (including hidden ones) and prints a table.
import ast, base64, contextlib, importlib.util, io, json, os, signal, sys, time, traceback, zlib
_P = json.loads(zlib.decompress(base64.b64decode("eNq9W41z27ix/1dQZaYmczIjKUnbU873npP4rnmTr4nTtB1Lw4NESOKZIlmCiu1z/b+/3y4AEpRkJ871vWRGFkFgv3exu4Cue2nSG4veZjQ4LAdPen3RW6taYuh6902d1pmisY8vg0Eoykol6bxOi1xcpPVK1Kpap7nMhMwTUVebfC5rlQhVprpIlCYImdK6yB3YzCJM802N92MxHOExSReLdL7J6iuMPMaAyuuKvvfqZBC3SHlpkWwyFedyzXSVVTHL1PpJrDezdaq1nSWzrLhQSZyuy6KqCdFZL9+sy6veFG9nMs8VsXpmn+KsKEqatZCZVsS30nVcp2tVbOqYXowGNCqXBlSVRXUSZ0pWeZovCSGNWGHYx1lR1LquZFnSFEKki001Z6qLKl26qaWs6hEGIT51g+dFmrFkrnsrgIf0opJFMeH/f1VZqSot6pWshWVeLCuZ0KDhVgS/LKpiLexyN/rwlzCa5JP82E4XqRbzIq8h60ebPK31ox/oz4+PLFQ8l2ny4yMzG0RE4lUtErVIAVWQgMQCCie9aEHqSHgwfmhoq+W5wjc1yVvNCKM88QuGfonEyT+O37x/fXIqslTXxJIyYAmYFnpVXOQCiz5schF8TnUKqsJn4pTA1QLGpoX6rKorXgTWfpJptqmwspKpVoIejZVKAXhaLpW4qNIaDItFUYnnKh8LLa/EBZF7ITXeFvmyL3KCKhJYi1ilSYLpaV5uak3SM1rIrUjnRXnVPJC1kHztI1ubANS8pNEH4kWRyZl48f5vWshKCU0GWpGsciLlQIs3I/G+KvpCFyIr5nAqQITpaLHeQCgzRUKFiArylRSvs5TkMAP/K6hyIed1UUWEaRQNSLlSLDek/8Bg1oCVqcN6VSmZCF0qKAyzqmKzXGVXsJZsATcmKuZFRYKuFAm9FumCCMQju5Llw/oGCeX0/cnJy/in4xcf330QR4SdGJ7k80wCO6khOLmcq5IsJRxPcoF/JV6ZWTAoSFwlAWwx6Yu1Xro5wJsXJOQ8sSP0r1VuQHNbIHFeBpduLcKHt8ZpqKjmK290Af7TXNcyn6vgsm/eRx9Vrosq9JYzWlVvqlxcRgli5XwVhNG83OCT1RyEZrJiNsUrRndSVUXlQTEse7DyMpJaVpW8At0tH3olSxUsi7oPo8zpE/Z5NOnBshEhJz1HmYas602ZqYA4x/ww4qWWFpYp5rhJBCvsi8Wkd00Ab6BwbXCJa424oy5LNafgfe3NvwG6lrJ5Vui7KOuLqi6yo6E6/FNfSPv1L45eeNYFSG6Ilbq+AqOLrJBEGI0zzu4Lj5ml4Y9YujBfb2HHMGCnh/t4c6+YvQYBFJJqhLe0xmrQkWVBKBApyAr9dxf2nY+eIqlMORLmtCbNFw1wa8ik7yyzQoQwrLjow4qLPnzDU1UFibEgiLi1vKQ/cqaDpTgUF2EYEmwwmv6mhMLOJQbkfHtcpSsnDnTis8wQH0SQyWpJYVeRwYpr/BlHj5c3YUf36l8bmX2FVXadykwlovOELb0fskRBhKzryr6f9FgdLZA95uIZSDvn/9wsfNNg+mMrBxC340y+VBsIpJXxNsUwBqKWmW9hIBhf49Ufqg5pNAtDHWWkOmYXYXXQN32rPhihp5F2yQ79NApRWMBhFHOSFccdcpYIf6Q5i/XATTpwULuUxjqXJXby+v83NGdFru4ZlIHSWeUlmSTv7b2QM1vy3i49AWWkfU5cQp+eLnMd6ghgkChVHn1EpucZmCXwI8S3Td4ugLCzhdBQRDD53WVH8FBLIPti9u1yl18ldzPF+AXj+1axSxI7w9HdUGARSYNEB7NwX6zB8jbO7F3exImZ/cq67Xo2cdA30Q6Wnfu6cnAYgpe8FPF6A6dAdhMs8r54iHAKz3j48PyCvjlKTOr4AjuAWOSMeUEJKvGAHAdZ4BLuJXOkjNVys0ZSji1ElJmcq8hlnbzrkz8hLp61niVDzmclLSCMUzMT2RltHnmwTZB5TUtSGDIY1iEtVUhlVIX6LfgtLQOzhnF0DJzjSWNcNohcobCBLSaKEvx0kVJmSSlzy8uD61R8J4Y3z8RFUZ1TVi/ZeAWpD/loEy+tkEF8K2FK4tM80IpSQVD1tVJ+5osY0dVPufdk0lwzYj/UyL3BAQmIc92DZiWSd1NDEap3AFFRCYYCaa2JJc7D38i5mAMBLQGTJuumaoUTe0oOUF0czjYJIikNcw6OtDylAmpTwp1EfZHOFQGEhFLs0MXCejBodhLVpuTGyAL5Em3fBAErNngXiY/0Iq0wDGayA0JB+4JBRhJWSces6gFlk4AQobpcxAxGVS7afMGUknr/YuQotU1HLL9HwipRPBJ+2dCaZEymWJE3BCPf7qBEoPnhyELaCkUz1DTn7ZAc9cX5CNi68ZFJ7m8Ndjj5siiYThLEiMUw8oZZDKjYgoS22FvE4eebPkPsR3VRsN2g+E/qcTRa3CDLh5WpZ7bqaoyYjQlux4DEJqd6/tqKxq7jGhKFnVPxnn+MEpUlyuAFTGtJCawpMq+tnhgQecedDlrlS3inSpzC2gIHmkyKNaS9kDC/uJk44S7NRkuZFjF3GDpdjp9pBB4AytrmQcpBY5VmCXndnDojKJhT8gWa+T+n796C1pw9R9cJKGRfLa/qFUYOXwmHR/xgWxxxklY/ih9aHHiJZ3jlv3ms/nGn5IezeeU/FHtZZ+msGTJ/MBJt6jRrh4vm66+6aJsEpaxX/nKdLnPZLtNXequ94B4q7AszOT9vRi5MK0qbXoOep2AzV024w8B5Wh9yx0oMoz9FQ5vT6TH1idbUpetzSEF1YTcfF/xKmULKy0nucESLNINFu8dg0gPZRaWoArSdFmSi0cO/5ec59XB0kVFHpeAOABCmJWktemgzxb+efDiB51hZRO/xN4hjaoPFcRghcNFyZHWlrEDZJIdUIpocwVpVVQcDOAPSBwITAuC+3hf3gsQD7NX/kmNx8mQw4k7Y678f//M0fn789u3JS1BwPenFtmMYx8QMGHP6NI/qUs3tN+T55tu8WJegddK7meSv3717H7999/LkFOAC2Er0UwFfpS9/h+Eq8/VYX+Xz5sVr5JAvAMM8nSrv4SWSzPbpZ5XT/lxUJ5dl6LdYPppezJe6LLFEpbcOyMY2a8QbbFuq8VguFR0gvxtBmc08nq/U/FwHupqTimvZ3W8/GH+nbBibFcmxODfCWeslBPMMYZEyQNPjY7fR9UGjHU6GbCuRNqq6Mu1e3dmizCZ0Nt2XzdaVQmLHUoKVoL4God089PQK9fkl56HUlFO7+eGZJXsMa+EmsKV+TGFS83JbIhe5iTPXKqK/eXEzpu+YjeLnxhJoO9C0JUBeEQIz/GSrLU0yOptaQmeyGed9snZbDrdD441mYN74DP4Qm16pHScTR8oI300TuAplYpVJFAxgQDDzA6poKOlOZ9SHBzvnnLupLCLDtgi4Ke7JfC3r9SbzRxbcHUGoptgM4V/I7DwgZWxt3V6aTtONPZvSYLuqaFJZmhgxuePdDcy2VJAvB5InUVK+ptaCeYyADYKkzDFYI/NEOKJijoCvmVijiHC8f2/0dBHJJLE4wt3JrW68eZEus5TUTTjPDofTPQtb5UWbMqGse89av0jM7hbjT4h629xAodAVy9E23sF+61B7ZMkrrCDx/XdIcVuCgLYlBU90HQnss4CtpbcJ75uWMp/3Ffqx851t3rcMgukgR/sqqG+9iHwXwDS5nav9k/ZhbDerbZzs+JEsS5Ungev4mEWTngl5Jrz/V1de+7DwLvg8zd/ZXex4szzWtAeFpreytSAq7Lw3sn6DlHHHpk0MuidxIMwzSA8kJYkW1K3h32UBxkuMydMoOwTNi34tqDzlABt4eMJwf3T5jnYUCMaBuj7oiwMDxHkUOeAB8KHuXR5QK/Zmfz9xH/3UsthDvna0t8AeNCeIM5lrku18ZXcITgXbDYLbz0qiiP0H/w2fmWo0u5BX+tCcp6KSQV6YNoDIKC20Is+u+g4nKpd2qwmMFZgmkk2lKOm4EplaojS/oLRJ8MwoikJvM0NSJpKClGIAr1KuL40WAm/T/CNvjt4ubMi1my/M8N8i8PbSP4pOXuiZEDDc23RAgRZWPgQ/0OEe2wHk8PdoOC8cjuZIttUyKO8yHxsniovS5JPdbhtNN0625XpfwzBVlLCL/xYONpWsnt+IwGROFsPZYOob9x727yOCFutdzHNw22GZR7+NYZMmsbF3GMUwuTEZdaVWKucitskdedV/ln/L2c0dlfqiSgEmu4o5kw0AAoVuTNVUN6N/l6tDpjNRel6lXFWIEqYK/14KexLNE1J3fE0FG9xV0TzNh/XdDtOM2iqubo1QN+N7HdezQEVx3LxAyWeTTQKOTHPBm/qC8AAENLWg8lO5nI8qv4YFm5VeUMuEGlYUZbk3aY0On5SOeVl7Z4hUwQZDs/k4rSXfCpNOTPiERHnnI5T931wz1hua74S9RiYeNLUYIu12lUuVrKyWn8+aFNGxcuvM0dQria2kKAE/Es2Uxy5dh91jnBoOUVbIRAcBE/GIzIZ8glsRPYKHPYs6GYELdiifCKIlxp/gpMHdxCOx6xemYJzYyzUTvkmj20fTmDEmaytpskQaeFtABT78sxbalMi5pRRtHJ0yWHJBfWaomvJuQU/sJVM/nXUlnctnNfcdd9D6UYK7FgGLky6H6MBM93d5YyWu5bkA0WkRndbUPnn1zgnPNHki8yewT6evfj5+/eFN31bpdirCWczNXXcK7MWy7hUlkuVo4GjpVsd8CaZtVkV0n6pScD6jjACUbqdZjkZVp4SicmS++vjqzcmH+MPJ8et+Q91WmqNLRdbTbYVFNBpTb8Z0dqg7yUcmHkfe1S5jGm1jDmryQtUWvs1sF52FxQgJdUAf2wvhMWaePqPXXDBMjeHvssQ+pKqIekCxWUaRJ/wGwQ26vQnbdum0JKwhWueYmmhmeKQIzFGNI25dFOd8piCunT7G0YA6wC50WSzPpVZNh8j0QNqW2PPXJ4PBsCXgXmzcRrE9GGIy6SSGbogVVlE2CbpjP2rcegf0PkG5uEJ4Yc9kUXwSH6AAfzIYDMbTb3ZkvozY7ePwkH+/w8xxZ8XuRBO89M2r8MvKbWWVFMokTeayHXYXAnHj1+33ZGF5P5+c9MylP/LCZsto7gE2fNt7hF/2vaXnfMvbfMlAs7PsdT/b2HJVnplCkcFdHXRJvA17tNdQ0hCgUFmYwE7fKLZ/lpV2OKK0Vmsd8O2ZvLsrcFR15/50gCJBRrAI/YYXogQdQnJKQhi3kmfejsn6qw3iljEKylb4FkHuWNsKuBSe03yjfP3NzQZr4uGYgZw9HU8Rv/lcGNTaLrXg8DjpWdBusoeN3lI3mMqefWWKjbfm0IeGBtHg5j5nYrsXH75h1/m9O489nNuNyt8U0qwSzlrJkadS4r17d2MngreLTe5hfJyDdVbkS3cusx2zRZDm5pYZZ/N9KiAWshLuZDD0g4BFzocbWz3t38lwQzOl13vuqtxrL/lP0eLfM+DziuQr95AWVmPgBK+CFSfBbQe1ffF43/Zm09mpK8iqJrDdeytqFphc9YhKx6DNXe/KSjmozIoiC3YIM+8IVN2xXAJXd8A1K1yBcuuWwteaF8IVPCa6xTHVN3RQNnbHA1TumJPdZrOgM919N6JvO6Xr84l239w17fuXa/p0zEyUNNfGEehtwFaXcl1S4Jt6NxbNWFOAuR8l8LJGsWd0fHg4jPA57JvAiFAaDN3gyBvE9+GABh/3OQyE0/43AHKLpp3C0lHXby4kBWeMq/14Gg2m3smcOVqP+chFs8zyuIWylpcItKrUR1RW4d3Rn7sV/imkCmEMoig/HJqz9aq4ACqzYWHM/Z6BjldlxbZDVf86TRLU9uK7IduUIp+hlBRLOgX/klysPe/fVsF0752PlgV/bwA/qqSjDPHoEQnyzEvmTIOQ1LEd/ugCiEZ8CIbmlqwRGNz7B3D91BT4h8OtmENM07oR0UQqzRENtiep0rl/QD5Dl0qGdOd+wUiPzJrmQm5f8KQRZrHLEopwuzfMZ3ijnYMZJgciChp9uoYaqZVVlak8UGXI5zZu0t4Tme4dGSdnx4kqw70G6RmcWqhKUaO+NbNP4E5m5Ur2xVKu18258Cdu3XDJ+tnkYZ9JoJ8Gntqh0rTF41HMkQ/uDsODslkEPHH70k9B92hJwo2kWORMiHgoPp3p0bS7BEMUaPnPd4ZwTAwqPDC0Q34V7lzTZ3/85F+FpyaEze9vFYa5lOtEYs6v/duCWB51f+B0F7Swlaz3ywFA7YuEWlJHOxfmP7U3Hj8Ndq5GjwHKux1tJ3cu+7pV7YVoc3v9U//rjKGLzf+BwPftDwSGI0+ufjTnPG4reBMSCKCJ7Zaq27QBP6eP783dDbtHbKNzcS6mMitVOoZ7xb+pqthHAAU1L/AOqesyuIsM6GqxwW78Z6YmpM8hh4u+b5ooFhiqIKgmdWZGKVCR7fPvnGj/V/32R3fahPA0n2ebhFrO0n+nygMtarrbz9f+fqUfEVXhHvbdz/aa0PsVfMMvHw8G/jbzhPaY70PH0APUQfSbAspTrsxPu+h+It8lo8tr+C5r0RJ8twRJGzr4nqU3MhrtSq8hc77hn0RJDx3JJZMghgjt/G6RjJw3ZfMTK43iNRPN7/b2yAqVl2JRGdF/haCQKjylXeRoOAy/hkdMI/4eM5NPt7ikyqFFxD+zcz/SsrYQIJpnV0LJ+UqY820xU4inio8hNtw7ImXkKMqscexhE1sLa/VLFnHGSc/I5jw2JfIzIS/pwfDIOmQzd3q3RGwaxJCecN5FGVDj1X8hrwaph6zXZr8ibno3N/8LUTvc0g==")).decode())
_dir = os.path.join("/tmp", "usaaio_grader_" + _P["id"])
os.makedirs(_dir, exist_ok=True)
for _n, _s in _P["files"].items():
    if _n.endswith("::b64"):
        with open(os.path.join(_dir, _n[:-5]), "wb") as _f:
            _f.write(base64.b64decode(_s))
        continue
    with open(os.path.join(_dir, _n), "w", encoding="utf-8") as _f:
        _f.write(_s)
if _dir not in sys.path:
    sys.path.insert(0, _dir)
for _m in ("harness", "usaaio_grade", "grader", _P["meta"].get("module_name", "submission")):
    sys.modules.pop(_m, None)
import harness as _harness
_harness.SPEED_FACTOR = 1.0   # Colab is the reference machine: time limits apply as stated
import usaaio_grade as _G

def _usaaio_grade():
    meta = _P["meta"]
    mod = meta.get("module_name", "submission")
    path = os.path.abspath(mod + ".py")
    rows, static = [], []
    if not os.path.exists(path):
        print(f"{mod}.py not found: run the %%writefile cell above first.")
        return
    src = open(path, encoding="utf-8").read()
    static = _G.static_checks(src, meta)
    per = float(meta.get("test_timeout_s", 20))
    def alarm(*_):
        raise TimeoutError()
    use_alarm = hasattr(signal, "SIGALRM")
    if use_alarm:
        signal.signal(signal.SIGALRM, alarm)
    def limit(s):
        if use_alarm:
            signal.setitimer(signal.ITIMER_REAL, s)
    buf = io.StringIO()
    error = None
    if any(not s["ok"] and s["msg"].startswith("syntax") for s in static):
        error = "syntax error"
    else:
        try:
            with contextlib.redirect_stdout(buf):
                limit(per)
                spec = importlib.util.spec_from_file_location(mod, path)
                sub = importlib.util.module_from_spec(spec)
                sys.modules[spec.name] = sub
                spec.loader.exec_module(sub)
                limit(0)
        except TimeoutError:
            error = f"importing your file took over {per:.0f} s"
        except BaseException as e:
            limit(0)
            error = "your file failed to import: " + _G.friendly_error(e, path)
        entry = meta.get("entry")
        if not error and entry and not hasattr(sub, entry):
            error = f"your file does not define {entry}"
    if not error:
        gspec = importlib.util.spec_from_file_location("grader", os.path.join(_dir, "grader.py"))
        grader = importlib.util.module_from_spec(gspec)
        sys.modules["grader"] = grader
        gspec.loader.exec_module(grader)
        visible = set(getattr(grader, "EXAMPLES", []))
        for name, fn in [(n, f) for n, f in vars(grader).items() if n.startswith("test_") and callable(f)]:
            rec = {"name": name[5:].replace("_", " ") + (" (example)" if name in visible else ""), "passed": False, "msg": ""}
            t0 = time.perf_counter()
            try:
                with contextlib.redirect_stdout(buf):
                    limit(per)
                    fn(sub)
                    limit(0)
                rec["passed"] = True
            except TimeoutError:
                rec["msg"] = f"took longer than {per:.0f} s (infinite loop, or far too slow)"
            except _harness.Fail as e:
                limit(0)
                rec["msg"] = str(e)
            except BaseException as e:
                limit(0)
                rec["msg"] = "your code raised " + _G.friendly_error(e, path)
            rec["seconds"] = time.perf_counter() - t0
            rows.append(rec)
    print("Rule checks")
    for s in static:
        print(f"  {'PASS' if s['ok'] else 'FAIL'}  {s['msg']}")
    if error:
        print("\n" + error)
    if rows:
        w = max(len(r["name"]) for r in rows)
        print("\nTests")
        for r in rows:
            print(f"  {'PASS' if r['passed'] else 'FAIL'}  {r['name']:<{w}}  {r['seconds']:6.2f} s  {r['msg']}")
    out = buf.getvalue()
    if out.strip():
        print("\nPrinted output (last 2000 characters)\n" + out[-2000:])
    n_ok = sum(r["passed"] for r in rows)
    ok = not error and rows and n_ok == len(rows) and all(s["ok"] for s in static)
    print("\n" + (f"Accepted: all {len(rows)} tests passed and every rule is met." if ok else
                  f"Not accepted: {n_ok} of {len(rows)} tests passed" + ("" if all(s["ok"] for s in static) else ", and a rule check failed") + "."))

_usaaio_grade()
